# Survey ETL Pipeline Notes

## Purpose

This pipeline moves survey data from a Google Sheet into a CSV file so it can be used in later data-processing steps.

## ETL Meaning

- **Extract:** Read the survey data from the Google Sheet.
- **Transform:** Use clear custom column names and check that the number of names matches the number of data columns.
- **Load:** Save the prepared data as a CSV file in the `raw` folder.

## Folder Responsibilities

- **`raw`:** Stores the latest data received from the source. This data is kept close to the original incoming data.
- **`error`:** Stores a file when the pipeline cannot save it successfully. This helps us investigate failed runs.
- **`target`:** Will store cleaned and transformed data after the next pipeline stage. It is not used by the current code yet.
- **`archive`:** Will store older successful raw files so we can keep history and compare previous runs. It is not used by the current code yet.

## Current Flow

```text
Google Sheet -> load data -> apply custom names -> check column count -> raw CSV
                                                                           |
                                                                           v
                                                                    error if failed
```

## Important Details

The Google Sheet is loaded with its original column names first. The custom names are defined in the final export step, where they are applied before the CSV is saved.

The column-count check prevents incorrect labels. If the data has 11 columns, the custom list must also contain exactly 11 names.

## Future Flow

```text
Google Sheet -> raw -> clean and transform -> target
                    |
                    +-> error when a step fails
                    +-> archive older raw files
```

The current pipeline creates the raw file and handles failed saves. Cleaning, target creation, and archiving can be added as later stages.

In [ ]:
# Call Python's built-in print function to display text in the notebook output.
# The string inside quotation marks is the exact message that will be displayed.
print("Hello, World!")

In [ ]:
# Explain that this cell is checking whether the pipeline notebook can execute successfully.
# Call Python's built-in print function to write the pipeline greeting to the output area.
# The text inside quotation marks is the message that will appear after the cell runs.
print("Hello pipeline")

In [20]:
# Import pandas so we can work with the sheet as a table.
import pandas as pd

# Use the Google Sheet's CSV export link for the first worksheet.
sheet_url = "https://docs.google.com/spreadsheets/d/1OzDHmE7ts9aI2fmqPUTJYUCLTCGTtHuQBYF2tquoRrg/export?format=csv&gid=0"

# Read the sheet with its original column names.
sheet_data = pd.read_csv(sheet_url)

# Display the first five rows of the loaded sheet.
sheet_data.head()

,Timestamp,How comfortable are you with Python right now?,What best describes your background?,Which direction would you like the class to take?,Which Python topics would you like more of? (select all that apply),Which architecture topics interest you most? (select all that apply),How do you learn best? (select all that apply),Is the class meeting your expectations?,How is the pace?,How is the difficulty level?,How is the balance between theory and hands-on practice?
0,9/30/2026 17:38:50,I can write basic scripts,Career switcher,Move forward – architecture-level design with ...,"pandas / data manipulation, PySpark, Working w...","Batch vs. streaming pipelines, Data lake / war...",An end-to-end project we build together,4,Just right,Just right,Well balanced
1,9/30/2026 17:39:29,Comfortable with libraries like pandas,Career switcher,A mix – short basics refreshers alongside arch...,"PySpark, Working with SQL from Python, APIs an...","ETL vs. ELT patterns, Transformation tools (e....","An end-to-end project we build together, Real-...",3,Just right,Too hard,Too much theory
2,9/30/2026 18:04:44,Beginner – still learning the syntax,Career switcher,A mix – short basics refreshers alongside arch...,"Core Python (functions, OOP, error handling), ...","ETL vs. ELT patterns, Orchestration (e.g. Airf...","An end-to-end project we build together, Real-...",3,Just right,Too hard,Well balanced
3,9/30/2026 18:11:39,I can write basic scripts,Student,A mix – short basics refreshers alongside arch...,"PySpark, Working with SQL from Python, APIs an...","Batch vs. streaming pipelines, Data lake / war...","Live coding sessions, An end-to-end project we...",4,Just right,Just right,Well balanced
4,9/30/2026 19:25:51,I can write basic scripts,Front End Developer,A mix – short basics refreshers alongside arch...,"Core Python (functions, OOP, error handling), ...","Batch vs. streaming pipelines, Data lake / war...","Live coding sessions, An end-to-end project we...",4,Just right,Just right,Well balanced


In [21]:
# Use the loaded Google Sheet data as the pipeline DataFrame.
df = sheet_data

In [24]:
# Import Path to create folder and file paths.
from pathlib import Path

# Import shutil to move a failed file.
import shutil

# Set the main project folder.
project_folder = Path(r"F:\Big Data\Survey")

# Set the folder for successful raw files.
raw_folder = project_folder / "raw"

# Set the folder for files that fail.
error_folder = project_folder / "error"

# Create the raw folder if it is missing.
raw_folder.mkdir(exist_ok=True)

# Create the error folder if it is missing.
error_folder.mkdir(exist_ok=True)

# Set the CSV file name.
file_name = "survey_responses.csv"

# Set the file path inside the raw folder.
raw_file = raw_folder / file_name

# Set the file path inside the error folder.
error_file = error_folder / file_name

try:
    # Define the custom names for the 11 columns.
    custom_columns = [
        "response_time",
        "python_level",
        "background",
        "class_direction",
        "python_topics",
        "architecture_topics",
        "learning_style",
        "expectations_rating",
        "pace",
        "difficulty",
        "theory_practice_balance"
    ]

    # Check that there is one custom name for every DataFrame column.
    if len(df.columns) != len(custom_columns):
        raise ValueError(
            f"Column count mismatch: data has {len(df.columns)} columns, "
            f"but custom_columns has {len(custom_columns)} names."
        )

    # Apply the custom column names only before saving the raw CSV.
    df.columns = custom_columns

    # Show the custom names being used in the CSV.
    print(f"Custom columns: {list(df.columns)}")

    # Save the DataFrame as a CSV in the raw folder.
    df.to_csv(raw_file, index=True, header=False)

    # Confirm that the file was saved.
    print(f"File saved to: {raw_file}")
except Exception as error:
    # Check whether a partly written file exists.
    if raw_file.exists():
        # Move the failed file to the error folder.
        shutil.move(str(raw_file), str(error_file))

        # Confirm that the file was moved.
        print(f"File moved to error folder: {error_file}")

    # Display the reason for the failure.
    print(f"Save failed: {error}")

Custom columns: ['response_time', 'python_level', 'background', 'class_direction', 'python_topics', 'architecture_topics', 'learning_style', 'expectations_rating', 'pace', 'difficulty', 'theory_practice_balance']
File saved to: F:\Big Data\Survey\raw\survey_responses.csv
